# End-to-End Colab Notebook
Mount Drive, define all classes inline, and run Stage 1 and Stage 2 training without importing local Python files. Splits are read from preprocessed JSONs for train/test.

In [ ]:

import os
import sys
from pathlib import Path

use_colab = 'google.colab' in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    base_dir = Path('/content/drive/MyDrive/liveness_detection_vae')
else:
    base_dir = Path.cwd()

data_dir = base_dir / '20GBprocessed'
source_split = base_dir / 'data_split.json'
preproc_dir = base_dir / 'preprocessed'
train_split_file = preproc_dir / 'stage2_train.json'
test_split_file = preproc_dir / 'final_test.json'
runs_dir = base_dir / 'runs_colab'
runs_dir.mkdir(parents=True, exist_ok=True)

print('Base dir:', base_dir)
print('Data dir:', data_dir)
print('Preprocessed splits dir:', preproc_dir)
print('Runs dir:', runs_dir)


In [ ]:

import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.signal import butter, sosfiltfilt, filtfilt

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:

preproc_dir.mkdir(parents=True, exist_ok=True)
if ((not train_split_file.exists()) or (not test_split_file.exists())) and source_split.exists():
    with open(source_split) as f:
        split_data = json.load(f)
    stage2_train = split_data.get('stage2_train', split_data.get('train', {'real': [], 'fake': []}))
    final_test = split_data.get('final_test', split_data.get('test', {'real': [], 'fake': []}))
    with open(train_split_file, 'w') as f:
        json.dump({'real': stage2_train.get('real', []), 'fake': stage2_train.get('fake', [])}, f)
    with open(test_split_file, 'w') as f:
        json.dump({'real': final_test.get('real', []), 'fake': final_test.get('fake', [])}, f)
    print('Generated preprocessed split files:', train_split_file, 'and', test_split_file)
else:
    print('Using existing preprocessed split files:', train_split_file, 'and', test_split_file)


In [ ]:

class Config:
    def __init__(self, mode='full'):
        mode = mode.lower()
        assert mode in {'full', 'simple'}
        self.mode = mode
        self.data_dir = str(data_dir)
        self.save_root = runs_dir
        self.T_fixed = 300 if mode == 'full' else 150
        self.fps = 30
        self.fc_low = 2.0
        self.fc_high = 8.0
        self.filter_order = 4
        self.C_h = 48
        self.C_z = 12
        self.dilations = [1, 2, 4]
        self.batch_size = 64
        self.lr = 1e-3
        self.epochs_stage1 = 20
        self.epochs_stage2 = 10
        self.val_split = 0.1
        self.beta_lf = self.beta_bp = self.beta_hf = 1.0
        self.device = device
        self.num_workers = 2 if use_colab else 4
        self.pin_memory = True
        if mode == 'full':
            self.use_velocity = True
            self.use_acceleration = True
            self.use_angle = True
            self.use_angle_rate = True
            self.F_dim = 8
            self.save_dir_stage1 = self.save_root / 'stage1_pretrain_full'
        else:
            self.use_velocity = True
            self.use_acceleration = False
            self.use_angle = False
            self.use_angle_rate = False
            self.F_dim = 4
            self.save_dir_stage1 = self.save_root / 'stage1_pretrain_simple'
        self.K_landmarks = 40
        self.C_in_per_band = self.K_landmarks * self.F_dim
        self.save_dir_stage1.mkdir(parents=True, exist_ok=True)


In [ ]:

# Filtering and feature helpers

def central_diff(arr):
    d = np.empty_like(arr)
    d[1:-1] = (arr[2:] - arr[:-2]) / 2.0
    d[0] = arr[1] - arr[0]
    d[-1] = arr[-1] - arr[-2]
    return d


def second_diff(arr):
    dd = np.empty_like(arr)
    dd[1:-1] = arr[2:] - 2 * arr[1:-1] + arr[:-2]
    dd[0] = arr[1] - arr[0]
    dd[-1] = arr[-1] - arr[-2]
    return dd


class ButterworthFilterBank:
    def __init__(self, fps=30, fc_low=2.0, fc_high=8.0, order=4):
        nyq = fps / 2.0
        wn_low = fc_low / nyq
        wn_high = fc_high / nyq
        self.sos_lf = butter(order, wn_low, btype='lowpass', output='sos')
        self.sos_bp = butter(order, [wn_low, wn_high], btype='bandpass', output='sos')
        self.sos_hf = butter(order, wn_high, btype='highpass', output='sos')

    def apply(self, x):
        T, K, D = x.shape
        x_flat = x.transpose(1, 2, 0).reshape(K * D, T)
        x_lf = sosfiltfilt(self.sos_lf, x_flat, axis=1).reshape(K, D, T).transpose(2, 0, 1)
        x_bp = sosfiltfilt(self.sos_bp, x_flat, axis=1).reshape(K, D, T).transpose(2, 0, 1)
        x_hf = sosfiltfilt(self.sos_hf, x_flat, axis=1).reshape(K, D, T).transpose(2, 0, 1)
        return x_lf, x_bp, x_hf


In [ ]:

class LipLivenessBandDataset(Dataset):
    def __init__(self, data_dir, cfg: Config, random_crop=False, ref_frames=10):
        self.files = sorted(Path(data_dir).glob('*.npz'))
        if not self.files:
            raise FileNotFoundError(f'No npz under {data_dir}')
        self.cfg = cfg
        self.random_crop = random_crop
        self.ref_frames = ref_frames
        self.filter_bank = ButterworthFilterBank(cfg.fps, cfg.fc_low, cfg.fc_high, cfg.filter_order)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        lips_outer = data['lips_outer']
        lips_inner = data['lips_inner']
        landmarks = np.concatenate([lips_outer[:, :-1, :], lips_inner[:, :-1, :]], axis=1)
        h, w = data['size']
        landmarks = landmarks.copy()
        landmarks[..., 0] /= w + 1e-8
        landmarks[..., 1] /= h + 1e-8
        landmarks = self._procrustes(landmarks)
        x_lf, x_bp, x_hf = self.filter_bank.apply(landmarks)
        feats_lf = self._features(x_lf)
        feats_bp = self._features(x_bp)
        feats_hf = self._features(x_hf)
        feats_lf = self._crop_or_pad(feats_lf)
        feats_bp = self._crop_or_pad(feats_bp)
        feats_hf = self._crop_or_pad(feats_hf)
        return (torch.from_numpy(feats_lf).float().transpose(0, 1),
                torch.from_numpy(feats_bp).float().transpose(0, 1),
                torch.from_numpy(feats_hf).float().transpose(0, 1))

    def _procrustes(self, landmarks):
        T, K, _ = landmarks.shape
        ref = landmarks[: min(self.ref_frames, T)].mean(axis=0)
        normed = np.empty_like(landmarks)
        for t in range(T):
            X, Y = landmarks[t], ref
            Xc, Yc = X - X.mean(axis=0), Y - Y.mean(axis=0)
            var = (Xc**2).sum() / Xc.shape[0] + 1e-12
            U, S, Vt = np.linalg.svd((Yc.T @ Xc) / Xc.shape[0])
            R = U @ Vt
            if np.linalg.det(R) < 0:
                Vt[-1, :] *= -1
                R = U @ Vt
            s = S.sum() / var
            tvec = Y.mean(axis=0) - s * (R @ X.mean(axis=0))
            normed[t] = s * (X @ R.T) + tvec
        return normed

    def _features(self, pos):
        fps = self.cfg.fps
        feats = [pos, central_diff(pos) * fps]
        if self.cfg.use_acceleration:
            feats.append(second_diff(pos) * (fps ** 2))
        if self.cfg.use_angle or self.cfg.use_angle_rate:
            v = np.roll(pos, -1, axis=1) - pos
            ang = np.arctan2(v[..., 1], v[..., 0])[..., None]
            if self.cfg.use_angle:
                feats.append(ang)
            if self.cfg.use_angle_rate:
                feats.append(central_diff(ang) * fps)
        return np.concatenate(feats, axis=2)

    def _crop_or_pad(self, feats):
        T, K, C = feats.shape
        target = self.cfg.T_fixed
        if T > target:
            if self.random_crop:
                start = np.random.randint(0, T - target + 1)
                feats = feats[start:start+target]
            else:
                start = (T - target) // 2
                feats = feats[start:start+target]
        elif T < target:
            pad = np.zeros((target, K, C), dtype=feats.dtype)
            pad[:T] = feats
            feats = pad
        return feats.reshape(target, K * C)


In [ ]:

class Stage2Dataset(Dataset):
    def __init__(self, split_json_path, split_name, cfg: Config, random_crop=True, base_dir=None):
        with open(split_json_path) as f:
            split_data = json.load(f)
        if split_name is None and 'real' in split_data and 'fake' in split_data:
            entries = split_data
        else:
            entries = split_data[split_name]
        self.files = []
        self.labels = []
        self.cfg = cfg
        self.random_crop = random_crop
        self.base_dir = Path(base_dir) if base_dir else None
        for fpath in entries.get('real', []):
            self.files.append(self._resolve(fpath))
            self.labels.append(0)
        for fpath in entries.get('fake', []):
            self.files.append(self._resolve(fpath))
            self.labels.append(1)
        self.fc_low = cfg.fc_low
        self.fc_high = cfg.fc_high
        self.filter_order = cfg.filter_order
        self.fps = cfg.fps

    def _resolve(self, p):
        path = Path(p)
        if path.exists():
            return str(path)
        if self.base_dir:
            candidate = self.base_dir / path.name
            if candidate.exists():
                return str(candidate)
        raise FileNotFoundError(p)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        lips_outer = data['lips_outer']
        lips_inner = data['lips_inner']
        landmarks = np.concatenate([lips_outer[:, :-1, :], lips_inner[:, :-1, :]], axis=1)
        h, w = data['size']
        landmarks[..., 0] /= w + 1e-8
        landmarks[..., 1] /= h + 1e-8
        feats = self._build_features(landmarks)
        feats = self._crop_or_pad(feats)
        x_lf, x_bp, x_hf = self._apply_filters(feats)
        return (torch.from_numpy(x_lf).float(),
                torch.from_numpy(x_bp).float(),
                torch.from_numpy(x_hf).float(),
                torch.tensor(self.labels[idx], dtype=torch.long))

    def _build_features(self, landmarks):
        T, N, _ = landmarks.shape
        fps = self.fps
        pos = landmarks
        vel = np.zeros_like(pos)
        vel[1:] = np.diff(pos, axis=0) * fps
        feats = [pos, vel]
        if self.cfg.use_acceleration:
            acc = np.zeros_like(pos)
            acc[1:] = np.diff(vel, axis=0) * fps
            feats.append(acc)
        if self.cfg.use_angle or self.cfg.use_angle_rate:
            ang = np.arctan2(landmarks[:, :, 1], landmarks[:, :, 0])[..., None]
            if self.cfg.use_angle:
                feats.append(ang)
            if self.cfg.use_angle_rate:
                ang_rate = np.zeros_like(ang)
                ang_rate[1:] = np.diff(ang, axis=0) * fps
                feats.append(ang_rate)
        return np.concatenate(feats, axis=-1)

    def _crop_or_pad(self, feats):
        T, N, C = feats.shape
        target = self.cfg.T_fixed
        if T > target:
            start = np.random.randint(0, T - target + 1) if self.random_crop else (T - target) // 2
            feats = feats[start:start+target]
        elif T < target:
            pad = np.zeros((target, N, C), dtype=feats.dtype)
            pad[:T] = feats
            feats = pad
        return feats

    def _apply_filters(self, feats):
        T, N, C = feats.shape
        b_lf, a_lf = butter(self.filter_order, self.fc_low / (self.fps / 2), btype='low')
        b_bp, a_bp = butter(self.filter_order, [self.fc_low / (self.fps / 2), self.fc_high / (self.fps / 2)], btype='band')
        b_hf, a_hf = butter(self.filter_order, self.fc_high / (self.fps / 2), btype='high')
        x_lf = np.zeros((T, N, C))
        x_bp = np.zeros((T, N, C))
        x_hf = np.zeros((T, N, C))
        for n in range(N):
            for c in range(C):
                s = feats[:, n, c]
                x_lf[:, n, c] = filtfilt(b_lf, a_lf, s)
                x_bp[:, n, c] = filtfilt(b_bp, a_bp, s)
                x_hf[:, n, c] = filtfilt(b_hf, a_hf, s)
        x_lf = x_lf.transpose(1, 2, 0).reshape(N * C, T)
        x_bp = x_bp.transpose(1, 2, 0).reshape(N * C, T)
        x_hf = x_hf.transpose(1, 2, 0).reshape(N * C, T)
        x_lf = (x_lf - x_lf.mean()) / (x_lf.std() + 1e-8)
        x_bp = (x_bp - x_bp.mean()) / (x_bp.std() + 1e-8)
        x_hf = (x_hf - x_hf.mean()) / (x_hf.std() + 1e-8)
        return x_lf, x_bp, x_hf


In [ ]:

# Model
class TCNBlock(nn.Module):
    def __init__(self, c_in, c_out, dilation=1, k=3):
        super().__init__()
        pad = dilation * (k - 1) // 2
        self.conv = nn.Conv1d(c_in, c_out, k, padding=pad, dilation=dilation)
        self.gn = nn.GroupNorm(1, c_out)
        self.act = nn.SiLU()
        self.res = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x):
        y = self.act(self.gn(self.conv(x)))
        return y + self.res(x)


class SingleBandVAE(nn.Module):
    def __init__(self, c_in, c_h=48, c_z=12, dilations=None):
        super().__init__()
        dilations = dilations or [1, 2, 4]
        self.enc_inp = nn.Conv1d(c_in, c_h, 1)
        self.encoder = nn.Sequential(*[TCNBlock(c_h, c_h, d) for d in dilations])
        self.enc_out = nn.Conv1d(c_h, c_z * 2, 1)
        self.dec_inp = nn.Conv1d(c_z, c_h, 1)
        self.decoder = nn.Sequential(*[TCNBlock(c_h, c_h, d) for d in reversed(dilations)])
        self.dec_out = nn.Conv1d(c_h, c_in, 1)

    def encode(self, x):
        h = self.encoder(self.enc_inp(x))
        mu, logvar = torch.chunk(self.enc_out(h), 2, dim=1)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder(self.dec_inp(z))
        return self.dec_out(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


class BandSplitVAE(nn.Module):
    def __init__(self, c_in_per_band, c_h=48, c_z=12, dilations=None):
        super().__init__()
        self.vae_lf = SingleBandVAE(c_in_per_band, c_h, c_z, dilations)
        self.vae_bp = SingleBandVAE(c_in_per_band, c_h, c_z, dilations)
        self.vae_hf = SingleBandVAE(c_in_per_band, c_h, c_z, dilations)
        self.fusion_weights = nn.Parameter(torch.ones(3))

    def forward(self, x_lf, x_bp, x_hf):
        recon_lf, mu_lf, logvar_lf = self.vae_lf(x_lf)
        recon_bp, mu_bp, logvar_bp = self.vae_bp(x_bp)
        recon_hf, mu_hf, logvar_hf = self.vae_hf(x_hf)
        weights = torch.softmax(self.fusion_weights, dim=0)
        fused = weights[0] * recon_lf + weights[1] * recon_bp + weights[2] * recon_hf
        return (
            {'lf': recon_lf, 'bp': recon_bp, 'hf': recon_hf},
            {'lf': mu_lf, 'bp': mu_bp, 'hf': mu_hf},
            {'lf': logvar_lf, 'bp': logvar_bp, 'hf': logvar_hf},
            fused,
        )


def kl_divergence(mu, logvar):
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    return kl.mean()


def band_split_vae_loss(recons, mus, logvars, targets, x_hat_fused=None, x_target_full=None, betas=None, alpha_fusion=0.0):
    betas = betas or {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
    total = 0.0
    loss_dict = {}
    for band in ['lf', 'bp', 'hf']:
        recon_loss = F.l1_loss(recons[band], targets[band])
        kl = kl_divergence(mus[band], logvars[band])
        band_loss = recon_loss + betas[band] * kl
        loss_dict[f'recon_{band}'] = recon_loss.item()
        loss_dict[f'kl_{band}'] = kl.item()
        total += band_loss
    if alpha_fusion > 0 and x_hat_fused is not None and x_target_full is not None:
        fusion = F.l1_loss(x_hat_fused, x_target_full)
        total = total + alpha_fusion * fusion
        loss_dict['fusion'] = fusion.item()
    loss_dict['total'] = total.item() if isinstance(total, torch.Tensor) else total
    return total, loss_dict


In [ ]:

# Stage 1 training

def train_epoch_stage1(model, loader, optimizer, cfg):
    model.train()
    total = recon_lf = recon_bp = recon_hf = kl_lf = kl_bp = kl_hf = 0.0
    n_batches = len(loader)
    for x_lf, x_bp, x_hf in loader:
        x_lf = x_lf.to(cfg.device)
        x_bp = x_bp.to(cfg.device)
        x_hf = x_hf.to(cfg.device)
        recons, mus, logvars, x_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': cfg.beta_lf, 'bp': cfg.beta_bp, 'hf': cfg.beta_hf}
        loss, ld = band_split_vae_loss(recons, mus, logvars, targets, x_fused, None, betas, 0.0)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += ld['total']; recon_lf += ld['recon_lf']; recon_bp += ld['recon_bp']; recon_hf += ld['recon_hf']
        kl_lf += ld['kl_lf']; kl_bp += ld['kl_bp']; kl_hf += ld['kl_hf']
    return {
        'total': total / n_batches,
        'recon_lf': recon_lf / n_batches,
        'recon_bp': recon_bp / n_batches,
        'recon_hf': recon_hf / n_batches,
        'kl_lf': kl_lf / n_batches,
        'kl_bp': kl_bp / n_batches,
        'kl_hf': kl_hf / n_batches,
    }


def validate_stage1(model, loader, cfg):
    model.eval()
    total = recon_lf = recon_bp = recon_hf = kl_lf = kl_bp = kl_hf = 0.0
    n_samples = 0
    with torch.no_grad():
        for x_lf, x_bp, x_hf in loader:
            x_lf = x_lf.to(cfg.device)
            x_bp = x_bp.to(cfg.device)
            x_hf = x_hf.to(cfg.device)
            recons, mus, logvars, x_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': cfg.beta_lf, 'bp': cfg.beta_bp, 'hf': cfg.beta_hf}
            loss, ld = band_split_vae_loss(recons, mus, logvars, targets, x_fused, None, betas, 0.0)
            bsz = x_lf.size(0)
            total += ld['total'] * bsz; recon_lf += ld['recon_lf'] * bsz; recon_bp += ld['recon_bp'] * bsz; recon_hf += ld['recon_hf'] * bsz
            kl_lf += ld['kl_lf'] * bsz; kl_bp += ld['kl_bp'] * bsz; kl_hf += ld['kl_hf'] * bsz
            n_samples += bsz
    return {
        'total': total / n_samples,
        'recon_lf': recon_lf / n_samples,
        'recon_bp': recon_bp / n_samples,
        'recon_hf': recon_hf / n_samples,
        'kl_lf': kl_lf / n_samples,
        'kl_bp': kl_bp / n_samples,
        'kl_hf': kl_hf / n_samples,
    }


def run_stage1(cfg: Config):
    dataset = LipLivenessBandDataset(cfg.data_dir, cfg, random_crop=True)
    train_size = int((1 - cfg.val_split) * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    model = BandSplitVAE(cfg.C_in_per_band, cfg.C_h, cfg.C_z, cfg.dilations).to(cfg.device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    best = float('inf')
    best_path = cfg.save_dir_stage1 / 'stage1_pretrained.pt'
    for epoch in range(1, cfg.epochs_stage1 + 1):
        start = time.time()
        tr = train_epoch_stage1(model, train_loader, opt, cfg)
        va = validate_stage1(model, val_loader, cfg)
        dur = (time.time() - start) / 60
        print(f'Epoch {epoch}/{cfg.epochs_stage1} train {tr['"'"'total'"'"']:.4f} val {va['"'"'total'"'"']:.4f} ({dur:.1f}m)')
        if va['total'] < best:
            best = va['total']
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': opt.state_dict(), 'val_metrics': va, 'config': vars(cfg)}, best_path)
            print('Saved', best_path)
    return best_path


In [ ]:

# Stage 2 margin variants

def margin_loss(real_loss, fake_loss, margin=0.5, lambda_margin=1.0):
    return real_loss + lambda_margin * torch.clamp(margin - fake_loss, min=0.0)


def validate_stage2_margin(model, loader, device, margin):
    model.eval()
    real_rec = fake_rec = 0.0
    n_real = n_fake = 0
    with torch.no_grad():
        for x_lf, x_bp, x_hf, labels in loader:
            x_lf = x_lf.to(device); x_bp = x_bp.to(device); x_hf = x_hf.to(device); labels = labels.to(device)
            recons, mus, logvars, x_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
            for i in range(labels.size(0)):
                sub_recons = {k: v[i:i+1] for k, v in recons.items()}
                sub_mus = {k: v[i:i+1] for k, v in mus.items()}
                sub_logvars = {k: v[i:i+1] for k, v in logvars.items()}
                sub_targets = {k: v[i:i+1] for k, v in targets.items()}
                sub_fused = x_fused[i:i+1]
                loss, _ = band_split_vae_loss(sub_recons, sub_mus, sub_logvars, sub_targets, sub_fused, None, betas, 0.0)
                if labels[i] == 0:
                    real_rec += float(loss); n_real += 1
                else:
                    fake_rec += float(loss); n_fake += 1
    real_mean = real_rec / max(n_real, 1)
    fake_mean = fake_rec / max(n_fake, 1)
    total = margin_loss(torch.tensor(real_mean, device=device), torch.tensor(fake_mean, device=device), margin)
    return {'total': float(total), 'real_rec': real_mean, 'fake_rec': fake_mean, 'separation': fake_mean - real_mean}


def train_epoch_stage2_margin(model, loader, opt, device, margin):
    model.train()
    total = real_rec = fake_rec = 0.0
    n_batches = len(loader)
    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device); x_bp = x_bp.to(device); x_hf = x_hf.to(device); labels = labels.to(device)
        opt.zero_grad()
        recons, mus, logvars, x_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
        real_mask = labels == 0
        fake_mask = labels == 1
        if real_mask.any():
            real_loss, real_dict = band_split_vae_loss({k: v[real_mask] for k, v in recons.items()}, {k: v[real_mask] for k, v in mus.items()}, {k: v[real_mask] for k, v in logvars.items()}, {k: v[real_mask] for k, v in targets.items()}, x_fused[real_mask], None, betas, 0.0)
        else:
            real_loss, real_dict = torch.zeros(1, device=device), {'total': 0.0}
        if fake_mask.any():
            fake_loss, fake_dict = band_split_vae_loss({k: v[fake_mask] for k, v in recons.items()}, {k: v[fake_mask] for k, v in mus.items()}, {k: v[fake_mask] for k, v in logvars.items()}, {k: v[fake_mask] for k, v in targets.items()}, x_fused[fake_mask], None, betas, 0.0)
        else:
            fake_loss, fake_dict = torch.zeros(1, device=device), {'total': 0.0}
        loss = margin_loss(real_loss, fake_loss, margin)
        loss.backward(); opt.step()
        total += loss.item(); real_rec += float(real_dict['total']); fake_rec += float(fake_dict['total'])
    return {'total': total / n_batches, 'real_rec': real_rec / n_batches, 'fake_rec': fake_rec / n_batches, 'separation': (fake_rec - real_rec) / n_batches}


def run_stage2_margin(cfg: Config, pretrained_path, margin=0.5, out_subdir='stage2_method1_margin'):
    save_dir = runs_dir / out_subdir
    save_dir.mkdir(parents=True, exist_ok=True)
    train_ds = Stage2Dataset(train_split_file, None, cfg, random_crop=True, base_dir=base_dir)
    val_ds = Stage2Dataset(test_split_file, None, cfg, random_crop=False, base_dir=base_dir)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    ckpt = torch.load(pretrained_path, map_location=cfg.device)
    model = BandSplitVAE(cfg.C_in_per_band, cfg.C_h, cfg.C_z, cfg.dilations).to(cfg.device)
    model.load_state_dict(ckpt['model_state_dict'])
    opt = optim.Adam(model.parameters(), lr=1e-4)
    best_sep = -float('inf')
    best_path = save_dir / 'best.pt'
    for epoch in range(1, cfg.epochs_stage2 + 1):
        start = time.time()
        tr = train_epoch_stage2_margin(model, train_loader, opt, cfg.device, margin)
        va = validate_stage2_margin(model, val_loader, cfg.device, margin)
        dur = (time.time() - start) / 60
        print(f'Epoch {epoch}/{cfg.epochs_stage2} train {tr['"'"'total'"'"']:.4f} val {va['"'"'total'"'"']:.4f} sep {va['"'"'separation'"'"']:.4f} ({dur:.1f}m)')
        if va['separation'] > best_sep:
            best_sep = va['separation']
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': opt.state_dict(), 'val_metrics': va, 'config': vars(cfg), 'margin': margin}, best_path)
            print('Saved', best_path)
    return best_path


In [ ]:

# Stage 2 discriminator variant
class LatentDiscriminator(nn.Module):
    def __init__(self, c_z, bands=3, hidden=128):
        super().__init__()
        input_dim = c_z * 2 * bands
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, z_lf, z_bp, z_hf):
        def pool(z):
            if z.dim() == 3:
                z_avg = z.mean(dim=2)
                z_max = z.max(dim=2)[0]
                return torch.cat([z_avg, z_max], dim=1)
            return z
        z = torch.cat([pool(z_lf), pool(z_bp), pool(z_hf)], dim=1)
        return self.net(z)


def train_epoch_stage2_disc(model, disc, loader, opt, device, lambda_cls=1.0):
    model.train(); disc.train()
    total = rec = cls = 0.0; n_batches = len(loader); correct = total_samples = 0
    crit = nn.BCEWithLogitsLoss()
    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device); x_bp = x_bp.to(device); x_hf = x_hf.to(device); labels = labels.to(device).float()
        opt.zero_grad()
        recons, mus, logvars, x_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
        loss_rec, ld = band_split_vae_loss(recons, mus, logvars, targets, x_fused, None, betas, 0.0)
        logits = disc(mus['lf'], mus['bp'], mus['hf']).squeeze(1)
        loss_cls = crit(logits, labels)
        loss = loss_rec + lambda_cls * loss_cls
        loss.backward(); opt.step()
        total += loss.item(); rec += ld['total']; cls += loss_cls.item()
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds == labels.long()).sum().item(); total_samples += labels.size(0)
    return {'total': total / n_batches, 'rec': rec / n_batches, 'cls': cls / n_batches, 'accuracy': correct / total_samples if total_samples else 0.0}


def validate_stage2_disc(model, disc, loader, device, lambda_cls=1.0):
    model.eval(); disc.eval(); total = rec = cls = 0.0; n_batches = len(loader); correct = total_samples = 0
    crit = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for x_lf, x_bp, x_hf, labels in loader:
            x_lf = x_lf.to(device); x_bp = x_bp.to(device); x_hf = x_hf.to(device); labels = labels.to(device).float()
            recons, mus, logvars, x_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
            loss_rec, ld = band_split_vae_loss(recons, mus, logvars, targets, x_fused, None, betas, 0.0)
            logits = disc(mus['lf'], mus['bp'], mus['hf']).squeeze(1)
            loss_cls = crit(logits, labels)
            loss = loss_rec + lambda_cls * loss_cls
            total += loss.item(); rec += ld['total']; cls += loss_cls.item()
            preds = (torch.sigmoid(logits) > 0.5).long()
            correct += (preds == labels.long()).sum().item(); total_samples += labels.size(0)
    return {'total': total / n_batches, 'rec': rec / n_batches, 'cls': cls / n_batches, 'accuracy': correct / total_samples if total_samples else 0.0}


def run_stage2_discriminator(cfg: Config, pretrained_path, lambda_cls=1.0):
    save_dir = runs_dir / 'stage2_method2_discriminator'
    save_dir.mkdir(parents=True, exist_ok=True)
    train_ds = Stage2Dataset(train_split_file, None, cfg, random_crop=True, base_dir=base_dir)
    val_ds = Stage2Dataset(test_split_file, None, cfg, random_crop=False, base_dir=base_dir)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    ckpt = torch.load(pretrained_path, map_location=cfg.device)
    model = BandSplitVAE(cfg.C_in_per_band, cfg.C_h, cfg.C_z, cfg.dilations).to(cfg.device)
    model.load_state_dict(ckpt['model_state_dict'])
    disc = LatentDiscriminator(cfg.C_z).to(cfg.device)
    opt = optim.Adam(list(model.parameters()) + list(disc.parameters()), lr=1e-4)
    best_acc = 0.0
    best_path = save_dir / 'best.pt'
    for epoch in range(1, cfg.epochs_stage2 + 1):
        start = time.time()
        tr = train_epoch_stage2_disc(model, disc, train_loader, opt, cfg.device, lambda_cls)
        va = validate_stage2_disc(model, disc, val_loader, cfg.device, lambda_cls)
        dur = (time.time() - start) / 60
        print(f'Epoch {epoch}/{cfg.epochs_stage2} train acc {tr['"'"'accuracy'"'"']:.4f} val acc {va['"'"'accuracy'"'"']:.4f} ({dur:.1f}m)')
        if va['accuracy'] > best_acc:
            best_acc = va['accuracy']
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'discriminator_state_dict': disc.state_dict(), 'optimizer_state_dict': opt.state_dict(), 'val_metrics': va, 'config': vars(cfg), 'lambda_cls': lambda_cls}, best_path)
            print('Saved', best_path)
    return best_path


In [ ]:

# Run switches
RUN_STAGE1 = False  # set True to pretrain
RUN_STAGE2 = 'none'  # options: 'none', 'margin', 'discriminator'
PRETRAINED_PATH = None  # set to existing checkpoint if skipping stage1
MARGIN = 0.5

cfg_full = Config('full')
pretrained_path = PRETRAINED_PATH
if RUN_STAGE1:
    pretrained_path = run_stage1(cfg_full)
else:
    if pretrained_path is None:
        candidate = cfg_full.save_dir_stage1 / 'stage1_pretrained.pt'
        pretrained_path = candidate if candidate.exists() else None
print('Using pretrained:', pretrained_path)

if RUN_STAGE2 == 'margin' and pretrained_path:
    run_stage2_margin(cfg_full, pretrained_path, margin=MARGIN)
elif RUN_STAGE2 == 'discriminator' and pretrained_path:
    run_stage2_discriminator(cfg_full, pretrained_path)
else:
    print('Stage2 skipped.')
